In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install librosa soundfile tqdm scikit-learn -q

Mounted at /content/drive


In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_curve
import matplotlib.pyplot as plt


BASE_DIR = "/content/drive/MyDrive/졸프_AI"
URBAN_DIR = os.path.join(BASE_DIR, "UrbanSound8K")

META_PATH = os.path.join(URBAN_DIR, "metadata", "UrbanSound8K.csv")
AUDIO_DIR = os.path.join(URBAN_DIR, "audio")

print("Metadata exists:", os.path.exists(META_PATH))
print("Audio dir exists:", os.path.exists(AUDIO_DIR))

Metadata exists: True
Audio dir exists: True


In [ ]:
meta = pd.read_csv(META_PATH)
meta.head()

print(meta.columns)
print(meta["class"].value_counts())

meta["label"] = (meta["class"] == "siren").astype(int)

print(meta[["class", "label"]].drop_duplicates().sort_values("label"))
print(meta["label"].value_counts())

## 사이렌 class만 1로, 나머지는 0으로

Index(['slice_file_name', 'fsID', 'start', 'end', 'salience', 'fold',
       'classID', 'class'],
      dtype='object')
class
dog_bark            1000
children_playing    1000
air_conditioner     1000
street_music        1000
jackhammer          1000
engine_idling       1000
drilling            1000
siren                929
car_horn             429
gun_shot             374
Name: count, dtype: int64
                class  label
0            dog_bark      0
1    children_playing      0
9            car_horn      0
22    air_conditioner      0
94       street_music      0
106          gun_shot      0
122     engine_idling      0
171        jackhammer      0
196          drilling      0
114             siren      1
label
0    7803
1     929
Name: count, dtype: int64


In [ ]:
train_meta = meta[meta["fold"].isin([1,2,3,4,5,6,7,8])].reset_index(drop=True)
val_meta   = meta[meta["fold"].isin([9])].reset_index(drop=True)
test_meta  = meta[meta["fold"].isin([10])].reset_index(drop=True)

print("train:", len(train_meta), train_meta["label"].value_counts().to_dict())
print("val  :", len(val_meta), val_meta["label"].value_counts().to_dict())
print("test :", len(test_meta), test_meta["label"].value_counts().to_dict())

## fold 1~8: train set/ fold 9: validation set/ fold 10: test set

train: 7079 {0: 6315, 1: 764}
val  : 816 {0: 734, 1: 82}
test : 837 {0: 754, 1: 83}


In [ ]:
# ============================================================
# FPGA-friendly FFT feature extraction
# Assumption:
#   - FPGA uses 48kHz I2S input
#   - FPGA decimates to 16kHz
#   - 512-point FFT IP
#   - 16 FFT blocks per one feature frame
#   - 16 frequency bands
#   - no Hann window
#   - no log1p
#   - no divider
#   - no StandardScaler
# ============================================================

SR = 16000

FFT_SIZE = 512
FFT_BLOCKS_PER_FRAME = 16

FRAME_LEN = FFT_SIZE * FFT_BLOCKS_PER_FRAME   # 8192 samples = 0.512 sec
HOP_LEN = FRAME_LEN                           # no overlap for FPGA simplicity

N_BANDS = 16
BINS_PER_BAND = 16                            # 256 valid bins / 16 bands

FEAT_PER_FRAME = 37
CONTEXT_FRAMES = 3
INPUT_DIM = FEAT_PER_FRAME * CONTEXT_FRAMES   # 111

EPS = 1e-8
NOISE_FLOOR = 2


def frame_signal_fpga(y, frame_len=FRAME_LEN, hop_len=HOP_LEN):
    frames = []

    if len(y) < frame_len:
        y = np.pad(y, (0, frame_len - len(y)))

    for start in range(0, len(y) - frame_len + 1, hop_len):
        frames.append(y[start:start + frame_len])

    return np.array(frames, dtype=np.float32)


def float_audio_to_s8(y):
    """
    FPGA에서 MIC_L_DATA[23:16]을 쓰는 것과 비슷하게
    Python에서는 audio를 signed int8 범위로 변환.
    """
    y = np.clip(y, -1.0, 1.0)
    s8 = np.round(y * 127.0)
    s8 = np.clip(s8, -128, 127).astype(np.int8)
    return s8


def float_audio_to_s16(y):
    """
    FFT IP 입력은 16-bit signed를 쓴다고 가정.
    """
    y = np.clip(y, -1.0, 1.0)
    s16 = np.round(y * 32767.0)
    s16 = np.clip(s16, -32768, 32767).astype(np.int16)
    return s16


def leading_one_pos_int(v):
    """
    Verilog leading-one detector와 맞추기 위한 함수.
    v > 0이면 floor(log2(v))와 유사.
    """
    v = int(v)

    if v <= 0:
        return 0

    return v.bit_length() - 1


def sat_int8(v):
    v = int(np.round(v))

    if v > 127:
        return 127
    elif v < -128:
        return -128
    else:
        return v


def log2_approx_feature(v):
    """
    np.log1p 대신 FPGA 친화적인 log2 근사.
    Verilog에서는 leading-one position으로 구현 가능.
    """
    if v <= 0:
        return np.int8(0)

    p = leading_one_pos_int(v)

    # scale factor는 경험적으로 6 사용
    # p가 커질수록 feature도 커짐
    feat = p * 6

    return np.int8(sat_int8(feat))


def normalize_band_by_shift(band_energy, total_energy):
    """
    division 없는 band normalization.
    기존 band_ratio = band / total 대신,
    total_energy의 MSB 위치를 보고 공통 shift를 적용.

    FPGA 구현:
      total_msb = leading_one(total_energy)
      shift = max(total_msb - 7, 0)
      norm_band = band_energy >> shift
      saturate to 127
    """
    total_energy = int(total_energy)

    if total_energy <= 0:
        return np.zeros_like(band_energy, dtype=np.int8)

    total_msb = leading_one_pos_int(total_energy)
    shift = max(total_msb - 7, 0)

    out = []

    for v in band_energy:
        vv = int(v) >> shift
        out.append(sat_int8(vv))

    return np.array(out, dtype=np.int8)


def zcr_feature_from_s8(s8):
    """
    Zero crossing rate feature.
    작은 잡음은 제외하기 위해 noise floor 적용.
    """
    zc = 0
    prev_valid = False
    prev_sign = 0

    for x in s8:
        ax = abs(int(x))

        if ax > NOISE_FLOOR:
            cur_sign = 1 if x < 0 else 0

            if prev_valid and cur_sign != prev_sign:
                zc += 1

            prev_sign = cur_sign
            prev_valid = True

    feat = (zc * 127) // len(s8)
    return np.int8(sat_int8(feat))


def mean_abs_feature_from_s8(s8):
    """
    RMS 대신 FPGA 구현이 쉬운 mean absolute value 사용.
    sum(abs(x)) / frame_len.
    """
    mean_abs = int(np.sum(np.abs(s8).astype(np.int32))) // len(s8)
    return np.int8(sat_int8(mean_abs))


def peak_band_feature(peak_idx):
    """
    0~15 band index를 0~120 정도로 scale.
    """
    return np.int8(sat_int8(int(peak_idx) * 8))


def extract_single_frame_features_fpga(frame_float):
    """
    8192-sample frame 하나에서 37개 int8 feature 생성.

    Feature 구성:
      0      : mean absolute amplitude
      1      : ZCR
      2      : peak band index
      3~18   : 16 log2-approx band energy
      19~34  : 16 shift-normalized band energy
      35     : delta energy, 나중에 file-level에서 추가
      36     : delta peak, 나중에 file-level에서 추가
    """

    # FPGA sample path와 맞춤
    s8 = float_audio_to_s8(frame_float)
    s16 = float_audio_to_s16(frame_float)

    # time-domain features
    mean_abs_feat = mean_abs_feature_from_s8(s8)
    zcr_feat = zcr_feature_from_s8(s8)

    # 16-band energy accumulator
    band_energy = np.zeros((N_BANDS,), dtype=np.int64)

    for blk in range(FFT_BLOCKS_PER_FRAME):
        start = blk * FFT_SIZE
        block = s16[start:start + FFT_SIZE].astype(np.float64)

        # FPGA FFT IP에 window를 넣지 않는다고 가정
        # 따라서 Python도 window 없음
        fft_out = np.fft.rfft(block, n=FFT_SIZE)

        # bins 0~255만 사용
        # rfft는 0~256까지 나오지만 Nyquist bin 256은 제외
        power = (np.real(fft_out[:256]) ** 2 + np.imag(fft_out[:256]) ** 2).astype(np.int64)

        for b in range(N_BANDS):
            s = b * BINS_PER_BAND
            e = s + BINS_PER_BAND
            band_energy[b] += np.sum(power[s:e])

    total_energy = int(np.sum(band_energy))

    peak_idx = int(np.argmax(band_energy))
    peak_feat = peak_band_feature(peak_idx)

    log_band_feat = np.array(
        [log2_approx_feature(v) for v in band_energy],
        dtype=np.int8
    )

    norm_band_feat = normalize_band_by_shift(band_energy, total_energy)

    # delta 2개는 extract_file_features_fpga에서 채움
    feat = np.concatenate([
        np.array([mean_abs_feat, zcr_feat, peak_feat], dtype=np.int8),
        log_band_feat,
        norm_band_feat,
        np.array([0, 0], dtype=np.int8)
    ])

    assert feat.shape[0] == FEAT_PER_FRAME

    return feat


def extract_file_features(path):
    """
    기존 extract_file_features 이름을 유지해서
    이후 make_dataset 코드를 그대로 쓸 수 있게 함.

    출력:
      shape = (num_frames, 111)
      dtype = int8
    """

    y, sr = librosa.load(path, sr=SR, mono=True)

    # 중요:
    # 파일 전체 max normalization 제거.
    # FPGA에서는 파일 전체 max를 알 수 없으므로 사용하면 안 됨.
    y = np.clip(y, -1.0, 1.0)

    frames = frame_signal_fpga(y)

    feats = []

    prev_energy = np.int8(0)
    prev_peak = np.int8(0)

    for frame in frames:
        f = extract_single_frame_features_fpga(frame)

        cur_energy = f[0]
        cur_peak = f[2]

        delta_energy = sat_int8(int(cur_energy) - int(prev_energy))
        delta_peak = sat_int8(int(cur_peak) - int(prev_peak))

        f[35] = np.int8(delta_energy)
        f[36] = np.int8(delta_peak)

        feats.append(f)

        prev_energy = cur_energy
        prev_peak = cur_peak

    feats = np.array(feats, dtype=np.int8)

    # 3-frame context 구성
    context_feats = []

    for i in range(len(feats)):
        stack = []

        for k in range(CONTEXT_FRAMES):
            idx = i - (CONTEXT_FRAMES - 1 - k)

            if idx < 0:
                stack.append(np.zeros((FEAT_PER_FRAME,), dtype=np.int8))
            else:
                stack.append(feats[idx])

        context_feats.append(np.concatenate(stack))

    context_feats = np.array(context_feats, dtype=np.int8)

    assert context_feats.shape[1] == INPUT_DIM

    return context_feats

## RMS: 소리가 얼마나 큰가 / band_ratio: 주파수 분포 정보
## 사이렌은 특정 주파수의 소리만 크게 나온다는 점을 이용

row = train_meta.iloc[0]

dummy_path = os.path.join(
    AUDIO_DIR,
    f"fold{row['fold']}",
    row["slice_file_name"]
)

print(dummy_path)
print(os.path.exists(dummy_path))

dummy_feat = extract_file_features(dummy_path)

print("dummy_feat shape:", dummy_feat.shape)
print("dummy_feat dtype:", dummy_feat.dtype)
print("min/max:", dummy_feat.min(), dummy_feat.max())
print(dummy_feat[0])

def make_dataset(df, name="train"):
    X_list = []
    y_list = []
    class_list = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=name):
        fold = row["fold"]
        filename = row["slice_file_name"]
        label = row["label"]
        cls = row["class"]

        path = os.path.join(AUDIO_DIR, f"fold{fold}", filename)

        try:
            feats = extract_file_features(path)
            labels = np.full((len(feats),), label, dtype=np.int32)
            classes = np.full((len(feats),), cls)

            X_list.append(feats)
            y_list.append(labels)
            class_list.append(classes)

        except Exception as e:
            print("Error:", path, e)

    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    classes = np.concatenate(class_list, axis=0)

    return X, y, classes

CACHE_DIR = os.path.join(BASE_DIR, "siren_cache_fpga_fft")
os.makedirs(CACHE_DIR, exist_ok=True)

train_cache = os.path.join(CACHE_DIR, "train_features.npz")
val_cache   = os.path.join(CACHE_DIR, "val_features.npz")
test_cache  = os.path.join(CACHE_DIR, "test_features.npz")

if os.path.exists(train_cache) and os.path.exists(val_cache) and os.path.exists(test_cache):
    train_data = np.load(train_cache, allow_pickle=True)
    val_data   = np.load(val_cache, allow_pickle=True)
    test_data  = np.load(test_cache, allow_pickle=True)

    X_train, y_train, cls_train = train_data["X"], train_data["y"], train_data["cls"]
    X_val, y_val, cls_val       = val_data["X"], val_data["y"], val_data["cls"]
    X_test, y_test, cls_test    = test_data["X"], test_data["y"], test_data["cls"]

else:
    X_train, y_train, cls_train = make_dataset(train_meta, "train")
    X_val, y_val, cls_val       = make_dataset(val_meta, "val")
    X_test, y_test, cls_test    = make_dataset(test_meta, "test")

    np.savez(train_cache, X=X_train, y=y_train, cls=cls_train)
    np.savez(val_cache, X=X_val, y=y_val, cls=cls_val)
    np.savez(test_cache, X=X_test, y=y_test, cls=cls_test)

print("X_train:", X_train.shape, X_train.dtype, X_train.min(), X_train.max())
print("X_val  :", X_val.shape, X_val.dtype, X_val.min(), X_val.max())
print("X_test :", X_test.shape, X_test.dtype, X_test.min(), X_test.max())


/content/drive/MyDrive/졸프_AI/UrbanSound8K/audio/fold5/100032-3-0-0.wav
True
dummy_feat shape: (1, 111)
dummy_feat dtype: int8
min/max: 0 127
[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   7   8  24 127 127 127 127 127 127 127 127 127 127 127 127 127
 127 127 127   0  35  27  44  24   3   0   0   0   0   0   0   0   0   0
   0   7  24]
X_train: (44540, 111) int8 -112 127
X_val  : (5197, 111) int8 -112 127
X_test : (5304, 111) int8 -112 127


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np

# ============================================================
# Raw score threshold sweep
# ============================================================
# sigmoid probability threshold가 아니라 raw score threshold를 확인.
# 최종 FPGA 권장값은 0이지만, 분포가 얼마나 벌어졌는지 확인하기 위한 표.
# ============================================================

def score_threshold_table(score, y, thresholds, name="data"):
    rows = []

    for th in thresholds:
        pred = (score >= th).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred).ravel()

        precision = precision_score(y, pred, zero_division=0)
        recall = recall_score(y, pred, zero_division=0)
        f1 = f1_score(y, pred, zero_division=0)

        rows.append({
            "name": name,
            "score_threshold": th,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

    return pd.DataFrame(rows)

score_thresholds = [-8.0, -6.0, -4.0, -2.0, -1.0, 0.0, 1.0, 2.0, 4.0, 6.0, 8.0]

val_table = score_threshold_table(val_score, y_val, score_thresholds, "validation")
test_table = score_threshold_table(test_score, y_test, score_thresholds, "test")

display(val_table)
display(test_table)


,name,score_threshold,TN,FP,FN,TP,precision,recall,f1
0,validation,-8.0,381,4255,0,561,0.116487,1.000000,0.208667
1,validation,-6.0,662,3974,0,561,0.123705,1.000000,0.220173
2,validation,-4.0,1239,3397,0,561,0.141738,1.000000,0.248285
3,validation,-2.0,2144,2492,0,561,0.183754,1.000000,0.310459
4,validation,-1.0,2947,1689,1,560,0.249000,0.998217,0.398577
5,validation,0.0,3760,876,21,540,0.381356,0.962567,0.546282
6,validation,1.0,4221,415,94,467,0.529478,0.832442,0.647263
7,validation,2.0,4527,109,210,351,0.763043,0.625668,0.687561
8,validation,4.0,4634,2,539,22,0.916667,0.039216,0.075214
9,validation,6.0,4636,0,561,0,0.000000,0.000000,0.000000


,name,score_threshold,TN,FP,FN,TP,precision,recall,f1
0,test,-8.0,39,4710,32,523,0.099943,0.942342,0.180719
1,test,-6.0,184,4565,64,491,0.097112,0.884685,0.175013
2,test,-4.0,887,3862,109,446,0.103528,0.803604,0.183426
3,test,-2.0,1777,2972,165,390,0.116002,0.702703,0.199132
4,test,-1.0,2921,1828,186,369,0.167956,0.664865,0.268169
5,test,0.0,3897,852,221,334,0.281619,0.601802,0.383688
6,test,1.0,4334,415,290,265,0.389706,0.477477,0.429150
7,test,2.0,4613,136,411,144,0.514286,0.259459,0.344910
8,test,4.0,4749,0,538,17,1.000000,0.030631,0.059441
9,test,6.0,4749,0,555,0,0.000000,0.000000,0.000000


In [ ]:
import os
import json
import numpy as np
import tensorflow as tf

# ============================================================
# 0. 기본 경로 및 최종 설정
# ============================================================

PARAM_DIR = os.path.join(BASE_DIR, "siren_params_fpga_fft")
COE_DIR   = os.path.join(BASE_DIR, "siren_coe_fpga_fft")

os.makedirs(PARAM_DIR, exist_ok=True)
os.makedirs(COE_DIR, exist_ok=True)

# Margin-loss raw score 모델의 최종 threshold
best_threshold = 0.0
best_score_threshold = 0.0
consecutive_need_count = 2

print("Final raw-score threshold:", best_score_threshold)
print("Consecutive need count:", consecutive_need_count)
print("PARAM_DIR:", PARAM_DIR)
print("COE_DIR:", COE_DIR)


# ============================================================
# 1. Quantization helper functions
# ============================================================

def quantize_symmetric_int8(x):
    """
    float weight를 signed int8로 symmetric quantization.
    q = round(x / scale)
    scale = max(abs(x)) / 127
    """
    x = np.array(x, dtype=np.float32)
    max_abs = np.max(np.abs(x))

    if max_abs < 1e-12:
        scale = 1.0
    else:
        scale = max_abs / 127.0

    q = np.round(x / scale)
    q = np.clip(q, -128, 127).astype(np.int8)

    return q, scale


def signed_to_twos_complement_hex(v, bit_width):
    """
    signed integer를 2's complement HEX 문자열로 변환.
    예:
      -1, 8bit  -> FF
      -1, 32bit -> FFFFFFFF
    """
    v = int(v)

    if v < 0:
        v = (1 << bit_width) + v

    mask = (1 << bit_width) - 1
    v = v & mask

    hex_width = bit_width // 4
    return f"{v:0{hex_width}X}"


def save_coe_hex(filename, data, bit_width):
    """
    Vivado COE용 HEX 저장.
    signed 값은 2's complement로 저장.
    """
    data = np.array(data).reshape(-1)

    with open(filename, "w") as f:
        f.write("memory_initialization_radix=16;\n")
        f.write("memory_initialization_vector=\n")

        for i, v in enumerate(data):
            h = signed_to_twos_complement_hex(v, bit_width)

            if i == len(data) - 1:
                f.write(f"{h};\n")
            else:
                f.write(f"{h},\n")

    print("saved:", filename, "length:", len(data), "bit_width:", bit_width)


def save_txt_signed(filename, data):
    """
    사람이 확인하기 쉬운 signed decimal txt 저장.
    """
    data = np.array(data).reshape(-1)
    np.savetxt(filename, data, fmt="%d")
    print("saved:", filename, "length:", len(data))


# ============================================================
# 2. Input feature quantization scale
# ============================================================
# FPGA-friendly feature는 이미 int8 feature이고,
# 학습 때 X_train_s = X_train_int8 / 127.0 으로 사용했음.
# 따라서 float 입력에서 int8 값 1은 1/127에 해당.
# ============================================================

input_scale = 1.0 / 127.0

print("\ninput_scale:", input_scale)

X_train_q = np.clip(np.round(X_train_s * 127.0), -128, 127).astype(np.int8)

print("X_train_q shape:", X_train_q.shape)
print("X_train_q min/max:", X_train_q.min(), X_train_q.max())

if "X_train" in globals():
    print("Original X_train dtype/min/max:", X_train.dtype, X_train.min(), X_train.max())


# ============================================================
# 3. Dense layer 추출
# ============================================================

_ = model(X_train_s[:1])

dense_layers = [
    layer for layer in model.layers
    if isinstance(layer, tf.keras.layers.Dense)
]

print("\nDense layers:")
for i, layer in enumerate(dense_layers):
    W, b = layer.get_weights()
    print(i, layer.name, W.shape, b.shape)

if len(dense_layers) != 3:
    print("Warning: Dense layer 개수가 3개가 아닙니다. model.summary()를 확인하세요.")


# ============================================================
# 4. Weight / Bias quantization
# ============================================================

q_params = {
    "model_type": "fpga_fft_feature_margin_score_mlp_post_training_quantization",
    "feature_type": "fpga_fft_int8_feature",
    "input_dim": int(X_train_s.shape[1]),
    "input_scale": float(input_scale),
    "threshold_type": "raw_score",
    "threshold_score_float": float(best_score_threshold),
    "threshold_score_int32": 0,
    "consecutive_need_count": int(consecutive_need_count),
    "standard_scaler_used": False,
    "feature_scaling": "X_s = X_int8 / 127.0",
    "final_output_activation": "linear_none",
    "decision_rule": "siren if raw final score >= 0",
    "layers": []
}

quantized_layers = []

prev_activation_scale = input_scale

for i, layer in enumerate(dense_layers):
    W, b = layer.get_weights()

    # ------------------------------------------------------------
    # Weight: signed int8
    # ------------------------------------------------------------
    W_q, W_scale = quantize_symmetric_int8(W)

    # ------------------------------------------------------------
    # Bias: signed int32
    # real bias ≈ b_q * prev_activation_scale * W_scale
    # ------------------------------------------------------------
    bias_scale = prev_activation_scale * W_scale

    if bias_scale < 1e-20:
        bias_scale = 1.0

    b_q = np.round(b / bias_scale).astype(np.int32)

    # ------------------------------------------------------------
    # 다음 activation scale 추정
    # hidden layer는 ReLU 이후 activation 기준
    # final layer는 raw score 분포 확인용 scale만 저장
    # ------------------------------------------------------------
    intermediate_model = tf.keras.Model(
        inputs=model.inputs[0],
        outputs=layer.output
    )

    act = intermediate_model.predict(
        X_train_s[:5000],
        batch_size=512,
        verbose=0
    )

    if i < len(dense_layers) - 1:
        act = np.maximum(act, 0)

    act_max_abs = np.max(np.abs(act))
    next_activation_scale = act_max_abs / 127.0 if act_max_abs > 1e-12 else 1.0

    quantized_layers.append({
        "W_q": W_q,
        "b_q": b_q,
        "W_scale": float(W_scale),
        "bias_scale": float(bias_scale),
        "prev_activation_scale": float(prev_activation_scale),
        "activation_scale": float(next_activation_scale)
    })

    q_params["layers"].append({
        "layer_index": int(i),
        "name": layer.name,
        "W_shape": list(W.shape),
        "b_shape": list(b.shape),
        "W_scale": float(W_scale),
        "bias_scale": float(bias_scale),
        "prev_activation_scale": float(prev_activation_scale),
        "activation_scale": float(next_activation_scale),
        "weight_bit_width": 8,
        "bias_bit_width": 32
    })

    print(f"\nLayer {i}: {layer.name}")
    print("  W float min/max:", np.min(W), np.max(W))
    print("  W_q min/max    :", W_q.min(), W_q.max())
    print("  b float min/max:", np.min(b), np.max(b))
    print("  b_q min/max    :", b_q.min(), b_q.max())
    print("  prev_act_scale :", prev_activation_scale)
    print("  W_scale        :", W_scale)
    print("  bias_scale     :", bias_scale)
    print("  act_scale next :", next_activation_scale)

    prev_activation_scale = next_activation_scale


# ============================================================
# 5. quant_params.json 저장
# ============================================================

quant_param_path = os.path.join(PARAM_DIR, "quant_params.json")

with open(quant_param_path, "w") as f:
    json.dump(q_params, f, indent=2)

print("\nSaved quant params:", quant_param_path)
print(json.dumps(q_params, indent=2))


# ============================================================
# 6. Weight / Bias COE 저장
# ============================================================

for i, ql in enumerate(quantized_layers):
    W_q = ql["W_q"]
    b_q = ql["b_q"]

    # W shape = [input_dim, output_dim]
    # Verilog에서 neuron 단위로 읽기 쉽게 transpose
    # 저장 순서: neuron0의 모든 weight, neuron1의 모든 weight, ...
    W_q_t = W_q.T

    weight_coe_path = os.path.join(COE_DIR, f"layer{i}_weight_int8.coe")
    bias_coe_path   = os.path.join(COE_DIR, f"layer{i}_bias_int32.coe")

    save_coe_hex(weight_coe_path, W_q_t.reshape(-1), bit_width=8)
    save_coe_hex(bias_coe_path, b_q.reshape(-1), bit_width=32)

    save_txt_signed(
        os.path.join(COE_DIR, f"layer{i}_weight_int8_signed.txt"),
        W_q_t.reshape(-1)
    )

    save_txt_signed(
        os.path.join(COE_DIR, f"layer{i}_bias_int32_signed.txt"),
        b_q.reshape(-1)
    )


# ============================================================
# 7. Feature scale info 저장
# ============================================================

feature_scale_info = {
    "feature_type": "fpga_fft_int8_feature",
    "standard_scaler_used": False,
    "training_input_scaling": "X_s = X_int8 / 127.0",
    "fpga_input_type": "signed int8",
    "input_scale": float(input_scale),
    "input_dim": int(X_train_s.shape[1]),
    "note": "No feature mean/std normalization is used in FPGA-friendly FFT feature model."
}

feature_scale_path = os.path.join(PARAM_DIR, "feature_scale_info.json")

with open(feature_scale_path, "w") as f:
    json.dump(feature_scale_info, f, indent=2)

print("\nSaved feature scale info:", feature_scale_path)


# ============================================================
# 8. Threshold / decision info 저장
# ============================================================

threshold_info = {
    "threshold_type": "raw_score",
    "threshold_score_float": float(best_score_threshold),
    "threshold_score_int32": 0,
    "consecutive_need_count": int(consecutive_need_count),
    "decision_rule_float": "siren_candidate = raw_score >= 0.0; final_siren = consecutive siren_candidate count >= consecutive_need_count",
    "fpga_decision_rule": "siren_detected <= (final_score >= THRESHOLD_SCORE), with THRESHOLD_SCORE = 0",
    "recommended_final_setting": "raw score threshold 0 + consecutive 2-frame decision",
    "feature_type": "fpga_fft_int8_feature"
}

threshold_path = os.path.join(PARAM_DIR, "threshold.json")

with open(threshold_path, "w") as f:
    json.dump(threshold_info, f, indent=2)

print("Saved threshold info:", threshold_path)


# ============================================================
# 9. 모델 weight 저장
# ============================================================

WEIGHT_PATH = os.path.join(PARAM_DIR, "siren_fpga_fft_margin_score_mlp_weights.weights.h5")
model.save_weights(WEIGHT_PATH)

print("Saved model weights:", WEIGHT_PATH)


# ============================================================
# 10. 최종 파일 목록 확인
# ============================================================

print("\nPARAM_DIR:", PARAM_DIR)
for f in sorted(os.listdir(PARAM_DIR)):
    print("  ", f)

print("\nCOE_DIR:", COE_DIR)
for f in sorted(os.listdir(COE_DIR)):
    print("  ", f)


Final raw-score threshold: 0.0
Consecutive need count: 2
PARAM_DIR: /content/drive/MyDrive/졸프_AI/siren_params_fpga_fft
COE_DIR: /content/drive/MyDrive/졸프_AI/siren_coe_fpga_fft

input_scale: 0.007874015748031496
X_train_q shape: (44540, 111)
X_train_q min/max: -112 127
Original X_train dtype/min/max: int8 -112 127

Dense layers:
0 dense0 (111, 32) (32,)
1 dense1 (32, 16) (16,)
2 dense2_output (16, 1) (1,)

Layer 0: dense0
  W float min/max: -1.6942545 1.3487828
  W_q min/max    : -127 101
  b float min/max: -0.047333777 0.041920226
  b_q min/max    : -451 399
  prev_act_scale : 0.007874015748031496
  W_scale        : 0.013340587
  bias_scale     : 0.00010504399
  act_scale next : 0.031078417

Layer 1: dense1
  W float min/max: -0.8903346 0.79666865
  W_q min/max    : -127 114
  b float min/max: -0.030251537 0.051676333
  b_q min/max    : -139 237
  prev_act_scale : 0.031078417
  W_scale        : 0.007010509
  bias_scale     : 0.00021787551
  act_scale next : 0.060489208

Layer 2: dense2

In [ ]:
## 학습 후 Verilog 파라미터 계산

import json
import os

PARAM_DIR = os.path.join(BASE_DIR, "siren_params_fpga_fft")

with open(os.path.join(PARAM_DIR, "quant_params.json"), "r") as f:
    qp = json.load(f)


def make_requant_params(acc_scale, act_scale, shift=20):
    ratio = acc_scale / act_scale
    mult = round(ratio * (1 << shift))
    return int(mult), int(shift), ratio


# Layer0
l0_acc_scale = qp["layers"][0]["bias_scale"]
l0_act_scale = qp["layers"][0]["activation_scale"]

# Layer1
l1_acc_scale = qp["layers"][1]["bias_scale"]
l1_act_scale = qp["layers"][1]["activation_scale"]

# Layer2 final raw-score accumulator scale
l2_acc_scale = qp["layers"][2]["bias_scale"]

L0_MUL, L0_SHIFT, L0_RATIO = make_requant_params(l0_acc_scale, l0_act_scale, shift=20)
L1_MUL, L1_SHIFT, L1_RATIO = make_requant_params(l1_acc_scale, l1_act_scale, shift=20)

# Margin-loss 모델은 sigmoid probability threshold가 아니라 raw score threshold를 사용.
# raw score threshold = 0이면 quantized final accumulator 기준도 0.
threshold_score_float = 0.0
THRESHOLD_SCORE = 0

print("L0 ratio:", L0_RATIO)
print("L1 ratio:", L1_RATIO)
print("threshold_score_float:", threshold_score_float)
print("l2_acc_scale:", l2_acc_scale)
print("THRESHOLD_SCORE:", THRESHOLD_SCORE)

print("\nVerilog parameters:")
print(f"parameter signed [31:0] L0_REQUANT_MUL   = 32'sd{L0_MUL};")
print(f"parameter        [5:0]  L0_REQUANT_SHIFT = 6'd{L0_SHIFT};")
print(f"parameter signed [31:0] L1_REQUANT_MUL   = 32'sd{L1_MUL};")
print(f"parameter        [5:0]  L1_REQUANT_SHIFT = 6'd{L1_SHIFT};")
print(f"parameter signed [31:0] THRESHOLD_SCORE  = 32'sd0;")

print("\nFPGA decision rule:")
print("siren_detected <= (final_score >= THRESHOLD_SCORE);")
print("// THRESHOLD_SCORE = 0")


L0 ratio: 0.0033799659132958044
L1 ratio: 0.003601890624607389
threshold_score_float: 0.0
l2_acc_scale: 0.0004981922684237361
THRESHOLD_SCORE: 0

Verilog parameters:
parameter signed [31:0] L0_REQUANT_MUL   = 32'sd3544;
parameter        [5:0]  L0_REQUANT_SHIFT = 6'd20;
parameter signed [31:0] L1_REQUANT_MUL   = 32'sd3777;
parameter        [5:0]  L1_REQUANT_SHIFT = 6'd20;
parameter signed [31:0] THRESHOLD_SCORE  = 32'sd0;

FPGA decision rule:
siren_detected <= (final_score >= THRESHOLD_SCORE);
// THRESHOLD_SCORE = 0
